In [3]:
import sys 
sys.path.append('../')

In [4]:
from src.connect import run_query

In [5]:
query = f"""

SELECT * 
FROM transa as t
"""

data = run_query(query)

In [6]:
data.head(5)

,id,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,1065008,8,CASH_OUT,158007.12,C424875646,0.0,0.0,C1298177219,474016.32,1618631.97,0
1,1100894,236,CASH_OUT,457948.30,C1342616552,0.0,0.0,C1323169990,2720411.37,3178359.67,0
2,1168189,37,CASH_IN,153602.99,C900876541,11160428.67,11314031.67,C608741097,3274930.56,3121327.56,0
3,1021833,331,CASH_OUT,49555.14,C177696810,10865.0,0.0,C462716348,0.0,49555.14,0
4,1178972,250,CASH_OUT,29648.02,C788941490,0.0,0.0,C1971700992,56933.09,86581.1,0


In [7]:
data.shape

(199999, 11)

In [8]:
data["isFraud"].value_counts(normalize=True)

isFraud
0    0.99859
1    0.00141
Name: proportion, dtype: float64

In [9]:
data.isnull().mean()

id                0.0
step              0.0
type              0.0
amount            0.0
nameOrig          0.0
oldbalanceOrg     0.0
newbalanceOrig    0.0
nameDest          0.0
oldbalanceDest    0.0
newbalanceDest    0.0
isFraud           0.0
dtype: float64

In [14]:
from src.utils.train import train_model, convert_categoricals, DROP_COLS, TARGET_COL
import joblib

def main(df):

    df = convert_categoricals(df.copy())
    df[TARGET_COL] = df[TARGET_COL].astype('int')
    X = df.drop(columns=DROP_COLS + [TARGET_COL])
    y = df[TARGET_COL]
    
    print(f"Features utilisées: {list(X.columns)}")
    
    best_model, scores = train_model(X, y)
    
    model_filename = "best_xgb_model_f1_.pkl"
    joblib.dump(best_model, model_filename)
    print(f"Modèle sauvegardé: {model_filename}")
    
    return best_model, scores

In [15]:
model, score = main(data)

Features utilisées: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
Démarrage de l'entraînement avec 5 folds...
Forme des données: (199999, 6)
Distribution des classes: {0: 199717, 1: 282}

=== Fold 1/5 ===
Scale pos weight: 706.96
F1 Score     : 0.3714
AUC Score    : 0.9468
Precision    : 0.9286
Recall       : 0.2321
✓ Nouveau meilleur modèle (F1: 0.3714)

=== Fold 2/5 ===
Scale pos weight: 706.96
F1 Score     : 0.4444
AUC Score    : 0.8940
Precision    : 1.0000
Recall       : 0.2857
✓ Nouveau meilleur modèle (F1: 0.4444)

=== Fold 3/5 ===
Scale pos weight: 710.11
F1 Score     : 0.4110
AUC Score    : 0.9204
Precision    : 0.9375
Recall       : 0.2632

=== Fold 4/5 ===
Scale pos weight: 710.11
F1 Score     : 0.4211
AUC Score    : 0.9145
Precision    : 0.8421
Recall       : 0.2807

=== Fold 5/5 ===
Scale pos weight: 706.96
F1 Score     : 0.4444
AUC Score    : 0.8967
Precision    : 1.0000
Recall       : 0.2857

=== RÉSULTATS FINAUX ===
Moyenne F1